# 2 — Solvers

**The concept:** the graph says what depends on what. The **solver** decides how to execute it.

| Solver | Use when | Cost |
|---|---|---|
| `DAGSolver` *(default)* | nothing consumes a value produced later | one pass |
| `HybridSolver` | anything cyclic — finds the loops itself | one pass, plus iteration on the cyclic parts only |
| `IterativeSolver` | you want manual control of order and convergence | sweeps **everything**, every iteration |

The short version: **reach for `HybridSolver` whenever there is feedback.**

In [1]:
from smartmdao import Pipeline, DAGSolver, HybridSolver, IterativeSolver, Solver

# `Solver` is the protocol the three of them satisfy: anything with a matching
# `solve()` can be dropped in, including one you write yourself.
for cls in (DAGSolver, IterativeSolver, HybridSolver):
    print(f"{cls.__name__:16} has solve(): {callable(getattr(cls, 'solve', None))}")

print()
print("protocol:", Solver.__name__)

DAGSolver        has solve(): True
IterativeSolver  has solve(): True
HybridSolver     has solve(): True

protocol: Solver


## DAGSolver — one pass, and it refuses a cycle

The default. It topologically sorts the graph and runs each step exactly once. If the graph has a
feedback loop there is no valid order, and rather than looping forever it **raises**.

In [2]:
linear = Pipeline(solver=DAGSolver())

@linear.step(outputs=["b"])
def first(a: float) -> float:
    return a * 2

@linear.step(outputs=["c"])
def second(b: float) -> float:
    return b + 1

linear.run(a=3.0)

{'a': 3.0, 'b': 6.0, 'c': 7.0}

In [3]:
cyclic = Pipeline(solver=DAGSolver())

@cyclic.step(outputs=["battery_kg"])
def battery(total_kg: float) -> float:
    return 0.25 * total_kg

@cyclic.step(outputs=["total_kg"])
def airframe(battery_kg: float) -> float:
    return 500.0 + battery_kg

try:
    cyclic.run(battery_kg=100.0)
except Exception as error:
    print(f"{type(error).__name__}: {error}")

Cycle detected in DAGSolver.


Pipeline execution failed: Cycle detected in pipeline. Use HybridSolver or IterativeSolver.


ValueError: Cycle detected in pipeline. Use HybridSolver or IterativeSolver.


That is the right behaviour: a loop needs an *iteration strategy*, and DAGSolver has none. It
says so instead of guessing.

## HybridSolver — decompose, then iterate only what needs it

`HybridSolver` finds the strongly connected components of the graph. Acyclic steps run **once**;
only the cyclic blocks are iterated to convergence. On a large model with one small loop, that is
the difference between iterating two disciplines and iterating forty.

In [4]:
snowball = Pipeline(solver=HybridSolver(max_iterations=100, tolerance=1e-8))

@snowball.step(outputs=["battery_kg"])
def battery(total_kg: float) -> float:
    return 0.25 * total_kg          # heavier aircraft needs more battery

@snowball.step(outputs=["total_kg"])
def airframe(battery_kg: float) -> float:
    return 500.0 + battery_kg       # ... which makes it heavier

result = snowball.run(battery_kg=100.0)
print(f"total_kg   = {result['total_kg']:.4f}")
print(f"battery_kg = {result['battery_kg']:.4f}")
print(f"converged in {result['convergence_reports'][0].iterations} iterations")

total_kg   = 666.6667
battery_kg = 166.6667
converged in 19 iterations


**Why seed `battery_kg` and not `total_kg`?** Inside a cyclic block `HybridSolver` runs steps
in **alphabetical order** for determinism. `airframe` sorts before `battery`, so `airframe` runs
first and reads `battery_kg` before anything has produced it.

This is genuinely surprising and it is covered properly in
[3 — Feedback loops](03-feedback-loops.ipynb). The rule to remember: **ask `analyze()`, do not
reason it out.**

## IterativeSolver — sweeps everything, in registration order

`IterativeSolver` ignores the dependency graph entirely. It sweeps every step, in the order you
registered them, until the values stop changing.

In [5]:
settling = Pipeline(solver=IterativeSolver(tolerance=1e-9, max_iterations=50))

@settling.step(outputs=["x"])
def relax(x: float) -> float:
    return 0.5 * x + 1.0

out = settling.run(x=0.0)
print(f"x -> {out['x']:.9f}   (fixed point of x = x/2 + 1 is 2)")

x -> 1.999999999   (fixed point of x = x/2 + 1 is 2)


**Registration order is load-bearing here, and getting it wrong is silent.** Because
`IterativeSolver` does not read the graph, a step registered before its producer simply sees the
previous value. See [15 — Pitfalls](15-pitfalls.ipynb).

Use it when you deliberately want manual control. Otherwise prefer `HybridSolver`, which derives
order from the graph and cannot make that mistake.

## The same steps, three solvers, three costs

`HybridSolver` runs the acyclic part once. `IterativeSolver` re-runs it on every sweep.

In [6]:
calls = {"acyclic": 0, "loop": 0}

def build(solver):
    calls["acyclic"] = calls["loop"] = 0
    p = Pipeline(solver=solver)

    @p.step(outputs=["scaled"])
    def preprocess(raw: float) -> float:
        calls["acyclic"] += 1
        return raw * 2

    @p.step(outputs=["y"])
    def loop_a(z: float, scaled: float) -> float:
        calls["loop"] += 1
        return (z + scaled) * 0.5

    @p.step(outputs=["z"])
    def loop_b(y: float) -> float:
        return y * 0.9 + 1.0

    return p

for solver in (HybridSolver(max_iterations=60), IterativeSolver(max_iterations=60)):
    p = build(solver)
    p.run(raw=1.0, z=0.0)
    print(f"{type(solver).__name__:16} acyclic step ran {calls['acyclic']:3} time(s), "
          f"loop step ran {calls['loop']:3}")

HybridSolver     acyclic step ran   1 time(s), loop step ran  20
IterativeSolver  acyclic step ran  20 time(s), loop step ran  20


That ratio is the entire argument for `HybridSolver`. When the acyclic step is a CFD run
rather than a multiplication, running it once instead of sixty times is the difference between a
study that finishes and one that does not.

---

**Next:** [3 — Feedback loops](03-feedback-loops.ipynb).